# P08 · Distancia al lugar de carga más cercano

> **¿Qué proporción de las paradas tiene un lugar de carga bip! a 300 metros o menos?**

GTFS Red Movilidad y los cinco Excel públicos de lugares de carga bip!.

Este cuaderno corre de principio a fin en Colab. La explicación completa, con las hojas de Excel dibujadas, está en [CUADERNO.md](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/cases/cobertura_de_carga/CUADERNO.md).

*Generado por `scripts/build_cuadernos.py`. No editar a mano: los cambios se pierden al regenerar.*


In [ ]:
# 1. Detectar el entorno y obtener una copia limpia del repositorio.
# Esta celda también funciona localmente si se abre el notebook dentro del repo.
import os
import platform
import subprocess
import sys
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules
if not Path('pyproject.toml').exists():
    destino_repo = Path('lab-dtpm')
    if not destino_repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab.git', str(destino_repo)], check=True)
    os.chdir(destino_repo)

ROOT = Path.cwd().resolve()
PAQUETES = ['pandas>=2.2', 'numpy>=1.26', 'openpyxl', 'XlsxWriter', 'matplotlib', 'polars>=1.0']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *PAQUETES], check=True)
print('Entorno:', 'Google Colab' if EN_COLAB else 'Python local/Jupyter')
print('Python:', platform.python_version())
print('Repositorio:', ROOT)


In [ ]:
# 2. Importar y registrar versiones: evita resultados imposibles de auditar.
import matplotlib
import numpy as np
import pandas as pd
import polars as pl

print({'pandas': pd.__version__, 'polars': pl.__version__, 'numpy': np.__version__, 'matplotlib': matplotlib.__version__})


## Alternativas para integrar los datos

La vía reproducible es clonar GitHub y ejecutar `scripts/fetch_data.py`, como hace la siguiente celda. Si la red institucional lo impide, consulte la [guía portable](https://github.com/vladimiracunadev-create/applied-data-analysis-and-assessment-lab/blob/main/docs/ejecucion-portable.md): explica carga desde el equipo, montaje de Google Drive, descarga directa y validación SHA-256. No cambie la fuente por datos simulados sin declararlo.


In [ ]:
# 3. Obtener las fuentes originales declaradas por el caso.
# Si la fuente externa falla, el error se conserva: no se sustituyen datos
# reales por sintéticos. Los CSV versionados permiten ejecutar el laboratorio
# comparativo, pero no reemplazan la adquisición del caso completo.
fuentes = ['gtfs-red-movilidad', 'bip-metro', 'bip-puntos', 'bip-centros-alto-estandar', 'bip-centros-estandar', 'bip-retail']
for fuente in fuentes:
    try:
        subprocess.run([sys.executable, 'scripts/fetch_data.py', fuente], check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            f'No se pudo obtener {fuente}. Revise la URL, la red y las alternativas de carga descritas arriba.'
        ) from exc


## 1 · El problema

Las paradas y los lugares de carga describen la misma ciudad, pero no comparten una
clave. La pregunta —qué proporción de las paradas tiene un lugar de carga a 300 metros—
obliga a declarar qué es una parada, cómo se mide una distancia sobre la Tierra y qué
significa «cerca».

El cruce comprende 12 497 paradas de embarque y 1 927 locales abiertos: más de 24
millones de pares. Una fórmula equivocada sigue produciendo un número razonable. Medir
grados como si fueran metros reduce la cobertura del 59,29 % al 54,69 % y asigna otro
local a 814 paradas. Además, la distancia en línea recta no es la distancia caminada;
por eso el cuaderno separa el dato calculado del supuesto de rodeo.


## 2 · La solución que se va a aplicar

Primero se fija el denominador y se audita la supuesta llave de los locales. Después se
calcula la distancia mínima con haversine por bloques, se describe su distribución y se
mide la sensibilidad al umbral y al factor de rodeo. Finalmente se confrontan dos
fórmulas y dos representaciones de coordenadas, se localizan zonas rezagadas y se
declara el límite que impide afirmar si el lugar cercano está abierto.


## Implementaciones comparables

La ruta principal conserva el caso completo en Pandas. El laboratorio comparable ejecuta Pandas y Polars en este runtime sobre el mismo CSV. R usa el mismo contrato y se valida en CI; el runtime Python estándar de Colab no ejecuta R de forma nativa.

#### Contrato común del ejercicio

- **Objetivo.** Medir cómo cambia la cobertura al variar el umbral de cercanía.
- **Fuente que leen las tres implementaciones.**
  [`resultados/cobertura_por_umbral.csv`](resultados/cobertura_por_umbral.csv), CSV UTF-8 versionado y producido
  por la resolución completa del caso. Los datos originales, su publicador, fecha,
  licencia, URL y huella se documentan en [`case.json`](case.json) y, cuando existe,
  `procedencia.json`. Este CSV es un **resultado reproducible**, no se presenta como
  una nueva fuente oficial.
- **Alcance.** curva de sensibilidad al umbral. Los tres scripts comparables no sustituyen la adquisición,
  limpieza ni controles completos de los pasos Pandas: aíslan el indicador para que
  una diferencia observada sea atribuible al motor.
- **Fórmula o regla.** cobertura(u) = 100 × #{paradas con distancia ≤ u} / #{paradas evaluables}. El denominador se mantiene fijo entre umbrales.
- **Interpretación.** La pendiente revela sensibilidad a una decisión metodológica; no demuestra demanda de nuevos puntos.
- **Error frecuente.** Cambiar simultáneamente umbral y denominador hace incomparables los puntos de la curva.

| Campo | Tipo observado | Función en el ejercicio |
| --- | --- | --- |
| `umbral_m` | `float64` | categoría/eje X |
| `paradas_cubiertas` | `int64` | campo auxiliar del contrato |
| `paradas_medidas` | `int64` | campo auxiliar del contrato |
| `cobertura_pct` | `float64` | medida/eje Y |
| `es_referencia` | `str` | campo auxiliar del contrato |

#### Qué cambia entre motores

| Dimensión | Pandas | Polars | R (`dplyr`/`readr`) |
| --- | --- | --- | --- |
| Modelo | DataFrame mutable por operaciones | expresiones columnares y plan optimizable | tubería declarativa sobre `tibble` |
| Nulos | `NaN`/`NA`; `groupby` exige decidir `dropna` | `null` separado de `NaN` | `NA`; muchas funciones exigen `na.rm` explícito |
| Orden | varias operaciones lo conservan, pero se ordena explícitamente | no se presupone tras agrupar | se fija con `arrange` |
| Gráfico | Matplotlib | Matplotlib tras extraer sólo X/Y | ggplot2 y su gramática de capas |
| Ventaja típica | ecosistema amplio y aprendizaje gradual | paralelismo, tipos y consultas grandes | estadística, comunicación y análisis reproducible |
| Límite aquí | el índice puede introducir alineaciones implícitas | añade una dependencia y otra semántica de tipos | requiere runtime y paquetes R independientes |

La equivalencia se valida sobre CSV normalizados con tolerancia numérica `1e-9`; no
se exige que la sintaxis ni los píxeles sean idénticos. Para ejecutar en un entorno
limpio consulte [`docs/ejecucion-portable.md`](../../docs/ejecucion-portable.md).

### Pandas + Matplotlib

```python
"""Laboratorio comparable Pandas + Matplotlib del caso cobertura_de_carga.

Lee el mismo contrato CSV que las versiones Polars y R. La resolución completa del
caso permanece en ``solution.py`` y en los pasos del cuaderno; este archivo aísla el
indicador central para que la comparación entre motores sea controlada.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

# Buscar la raíz desde el propio archivo evita rutas C:\... específicas del autor y
# funciona igual después de clonar el repositorio en Linux, Windows o Google Colab.
ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "cobertura_de_carga" / "resultados" / "cobertura_por_umbral.csv"
REQUIRED = {"umbral_m", "cobertura_pct"}


def analizar() -> pd.DataFrame:
    """Carga, valida y transforma el indicador; no modifica el archivo de entrada."""
    datos = pd.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # La expresión conserva explícitos filtro, agrupación, orden y selección para que
    # el estudiante pueda alinearlos con las expresiones de Polars y dplyr.
    resultado = datos.sort_values("umbral_m")[["umbral_m", "cobertura_pct"]]
    return resultado.reset_index(drop=True)


def graficar(resultado: pd.DataFrame, destino: Path) -> Path:
    """Representa exactamente las columnas comparadas y explicita sus unidades."""
    x = resultado["umbral_m"].tolist()
    y = resultado["cobertura_pct"].tolist()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Umbral de cercanía (m)")
    ax.set_ylabel("Paradas cubiertas (%)")
    ax.set_ylim(bottom=0)
    ax.set_title("La cobertura depende del umbral declarado")
    ax.grid(axis="y", alpha=0.25)
    fig.text(0.01, 0.01, "Fuente: Resultados reproducibles del caso P08 · Período: corte publicado", fontsize=8, color="#58677A")
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado.to_string(index=False))
    if args.salida:
        resultado.to_csv(args.salida, index=False)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `python cases/cobertura_de_carga/cuaderno/solucion_pandas.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/cobertura_de_carga/cuaderno/solucion_pandas.py', '--grafico', 'solucion_pandas-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_pandas-colab.png'))


### Polars + Matplotlib

```python
"""Laboratorio comparable Polars + Matplotlib del caso cobertura_de_carga.

Usa el mismo CSV, columnas, reglas y escala que Pandas y R. Polars expresa las
transformaciones como un plan de expresiones columnares; esa diferencia de método es
el objeto de aprendizaje, no una razón para cambiar la pregunta.
"""
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

ROOT = Path(__file__).resolve().parents[3] if "__file__" in globals() else Path.cwd()
SOURCE = ROOT / "cases" / "cobertura_de_carga" / "resultados" / "cobertura_por_umbral.csv"
REQUIRED = {"umbral_m", "cobertura_pct"}


def analizar() -> pl.DataFrame:
    """Carga y valida antes de transformar para producir un error explicativo."""
    datos = pl.read_csv(SOURCE)
    faltantes = REQUIRED.difference(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas requeridas en {SOURCE}: {sorted(faltantes)}")
    # select al final fija el contrato de salida y evita comparar columnas auxiliares.
    return datos.sort("umbral_m").select(["umbral_m", "cobertura_pct"])


def graficar(resultado: pl.DataFrame, destino: Path) -> Path:
    """Convierte sólo las series del gráfico; el análisis permanece en Polars."""
    x = resultado["umbral_m"].to_list()
    y = resultado["cobertura_pct"].to_list()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(x, y, marker="o", linewidth=2, color="#11253B")
    ax.set_xlabel("Umbral de cercanía (m)")
    ax.set_ylabel("Paradas cubiertas (%)")
    ax.set_ylim(bottom=0)
    ax.set_title("La cobertura depende del umbral declarado")
    ax.grid(axis="y", alpha=0.25)
    fig.text(
        0.01,
        0.01,
        "Fuente: Resultados reproducibles del caso P08 · Período: corte publicado",
        fontsize=8,
        color="#58677A",
    )
    fig.tight_layout(rect=(0, 0.05, 1, 1))
    fig.savefig(destino, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return destino


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--salida", type=Path, help="CSV normalizado para comparar motores")
    parser.add_argument("--grafico", type=Path, help="PNG de la visualización comparable")
    args = parser.parse_args()
    resultado = analizar()
    print(resultado)
    if args.salida:
        resultado.write_csv(args.salida)
    if args.grafico:
        print(graficar(resultado, args.grafico))


if __name__ == "__main__":
    main()
```

Ejecutar con `uv run --extra polars python cases/cobertura_de_carga/cuaderno/solucion_polars.py`.


In [ ]:
# Ejecutar la implementación publicada, no una copia pegada.
subprocess.run([sys.executable, 'cases/cobertura_de_carga/cuaderno/solucion_polars.py', '--grafico', 'solucion_polars-colab.png'], check=True)
from IPython.display import Image, display
display(Image('solucion_polars-colab.png'))


### R + ggplot2

```r
#!/usr/bin/env Rscript
# Laboratorio comparable R + dplyr/readr + ggplot2 del caso cobertura_de_carga.
#
# La resolución completa sigue en solution.py y en los pasos Pandas. Este script
# cambia el motor sobre el mismo contrato CSV para estudiar sintaxis y semántica sin
# cambiar datos, indicador ni escala gráfica.

required <- c("readr", "dplyr")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) {
  stop(
    "Faltan paquetes: ", paste(missing, collapse = ", "),
    ". Instálelos con install.packages(c('readr', 'dplyr', 'ggplot2'))."
  )
}

# Buscar pyproject.toml hacia arriba hace portable el script entre terminal, RStudio,
# Posit Cloud, IRkernel y un clon en Colab. LAB_ROOT permite una ruta explícita sin
# codificar la unidad o el directorio del desarrollador.
find_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "pyproject.toml"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("No se encontró la raíz del repositorio.")
    current <- parent
  }
}
root <- Sys.getenv("LAB_ROOT", unset = find_root())
source_csv <- file.path(root, "cases", "cobertura_de_carga", "resultados", "cobertura_por_umbral.csv")

# readr conserva nombres y tipos previsibles. La validación temprana produce un error
# que explica el contrato roto, en vez de fallar más tarde dentro de una gráfica.
datos <- readr::read_csv(source_csv, show_col_types = FALSE, progress = FALSE)
required_columns <- c("umbral_m", "cobertura_pct")
missing_columns <- setdiff(required_columns, names(datos))
if (length(missing_columns)) stop("Faltan columnas: ", paste(missing_columns, collapse = ", "))

# dplyr expresa la misma selección, filtro, agrupación y orden que las otras vías.
resultado <- datos |> dplyr::arrange(.data$umbral_m) |> dplyr::select(.data$umbral_m, .data$cobertura_pct)
print(resultado)

args <- commandArgs(trailingOnly = TRUE)
value_after <- function(flag) {
  pos <- match(flag, args)
  if (is.na(pos)) return(NULL)
  if (pos == length(args)) stop(flag, " necesita una ruta de salida")
  args[pos + 1]
}

csv_output <- value_after("--salida")
if (!is.null(csv_output)) readr::write_csv(resultado, csv_output)

plot_output <- value_after("--grafico")
if (!is.null(plot_output)) {
  if (!requireNamespace("ggplot2", quietly = TRUE)) {
    stop("Instale ggplot2 para generar el gráfico: install.packages('ggplot2')")
  }
  grafico <- ggplot2::ggplot(
    resultado,
    ggplot2::aes(x = .data[["umbral_m"]], y = .data[["cobertura_pct"]])
  ) +
    ggplot2::geom_line(ggplot2::aes(group = 1), linewidth = 1, colour = "#11253B") +
    ggplot2::geom_point(size = 2.5, colour = "#55D6BE") +
    ggplot2::labs(
      title = "La cobertura depende del umbral declarado",
      x = "Umbral de cercanía (m)", y = "Paradas cubiertas (%)",
      caption = "Fuente: Resultados reproducibles del caso P08 · Período: corte publicado"
    ) +
    ggplot2::theme_minimal(base_size = 11)
  ggplot2::ggsave(plot_output, grafico, width = 9, height = 5, dpi = 150)
  cat(plot_output, "\n")
}
```

Ejecutar con `Rscript cases/cobertura_de_carga/cuaderno/solucion.R`.


### Ejecutar R

Use `solucion.R` desde R/RStudio, Posit Cloud o Jupyter con IRkernel. También se incluye `solucion.Rmd`. En Colab, seleccione un runtime R compatible o use la instalación experimental documentada; se marca como compatibilidad documentada, no como ejecución validada.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cases.cobertura_de_carga import solution as p08
import pandas as pd

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 30)


## 3 · Los pasos


### Paso 1 · Fijar qué filas son paradas


El denominador contiene lugares donde sube gente, no estaciones, accesos ni nodos.

> ⚠️ Contar todo `stops.txt` agrega 5 989 elementos que no son paradas de embarque.


In [ ]:
resultados = p08.analizar()
resultados['paradas_por_tipo']


### Paso 2 · Comprobar la llave de los locales


Antes del cruce, se mide si `codigo` identifica un lugar de carga.

> ⚠️ En Metro el código es la línea: 143 filas comparten solo siete códigos.


In [ ]:
resultados['identificador']


### Paso 3 · Medir el vecino más cercano


Se evalúan 24 081 719 pares con haversine por bloques y se conserva el mínimo por parada.

> ⚠️ Crear la matriz completa consume memoria innecesaria; por bloques conserva el mismo mínimo.


In [ ]:
resultados['resumen']


### Paso 4 · Describir la distribución, no solo la media


Percentiles, mediana y máximo muestran una cola que la media oculta.

**En Excel:** `=PERCENTIL.INC(rango_distancia;0,5)`

Debe dar: **249,6 m de mediana**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ El máximo de 40 km pertenece al borde de cobertura de los archivos, no a una caminata plausible.


In [ ]:
resultados['distribucion']


### Paso 5 · Mostrar cuánto depende del umbral


La cobertura se publica junto a la curva de 200 a 1 000 metros.

**En Excel:** `=CONTAR.SI(rango_distancia;"<="&E2)/CONTAR(rango_distancia)`

Debe dar: **59,29 % a 300 m**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Elegir el umbral después de ver el resultado convierte el criterio en una meta móvil.


In [ ]:
resultados['cobertura_por_umbral']


### Paso 6 · Separar línea recta de distancia caminada


El factor de rodeo es un supuesto explícito y su efecto se expresa en puntos porcentuales.

**En Excel:** `=SUMAPRODUCTO(--(rango_distancia*factor<=300))/CONTAR(rango_distancia)`

Debe dar: **46,02 % con factor 1,3**

**En Google Sheets:** la misma fórmula, sin cambios.

> ⚠️ Decir 300 m caminando a partir de una línea geodésica sobrestima la cobertura.


In [ ]:
resultados['sensibilidad_rodeo']


### Paso 7 · Validar fórmula y representación


Dos controles independientes cuantifican cuánto cambia la clasificación de cada parada.

> ⚠️ Que el agregado cambie poco no significa que las mismas paradas queden cubiertas.


In [ ]:
pd.concat([resultados['formulas_comparadas'], resultados['representaciones_comparadas']], ignore_index=True)


### Paso 8 · Localizar zonas con menor cobertura


Una rejilla publicable sustituye la comuna que no existe en el GTFS.

> ⚠️ Asignar la comuna del local más cercano a la parada crea comunas falsas en los bordes.


In [ ]:
resultados['por_rejilla'].head(10)


### Paso 9 · Declarar lo que el dato no permite afirmar


Se distingue cercanía física de disponibilidad efectiva del punto de carga.

> ⚠️ Una parada cubierta espacialmente puede apuntar a un local cuyo horario no fue publicado.


In [ ]:
resultados['limite_por_horario']


## 4 · El resultado

La cobertura es 59,29 % en línea recta y 46,02 % con un rodeo supuesto de 1,3. Son dos respuestas a dos preguntas distintas; ninguna autoriza a afirmar que el local esté abierto.

## 5 · Qué queda aprendido

1. Un cruce espacial también tiene una llave: el criterio, el umbral y la fórmula declarados.
2. La sensibilidad debe acompañar al resultado cuando una decisión metodológica cambia más de diez puntos.
3. Un agregado estable puede esconder centenares de filas que cambian de clasificación.
4. La ausencia de comuna y horario limita qué recomendación es defendible.
